# Pulling Data in at the ZCTA level for our Restaurant Health Inspection Scores Dataset

Notebook opened: 4/27/25

In [24]:
#imports
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

Links to Census data:
- Using TIGER shapefiles for Census Tracts

In [25]:
#loading data

#geocoded restaurant health inspection data
df = pd.read_csv("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Output/healthinspections_2024_geocoded.csv")

/var/folders/81/w_61xz297rv4ggdktb58tlxm0000gn/T/ipykernel_96626/2128334811.py:4: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Output/healthinspections_2024_geocoded.csv")


In [26]:
#convert dataframe into GeoDataFrame
print(df.columns.tolist())

#Creating geometry column
geometry = gpd.points_from_xy(df.Longitude, df.Latitude)
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")  # WGS84


['INSPECTION_DATE', 'STORE_NAME', 'STREET_ADDRESS', 'CITY', 'ZIP5', 'SERVICE_DESCRIPTION', 'SCORE', 'GRADE', 'STREET_ADDRESS_LINE2', 'INSPDATE_YEAR', 'STATE', 'Latitude', 'Longitude', 'Accuracy Score', 'Accuracy Type', 'Number', 'Street', 'Unit Type', 'Unit Number', 'City', 'State', 'County', 'Zip', 'Country', 'Source']


In [27]:
# Reading in ZCTA boundaries
CAtract_gdf = gpd.read_file("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Raw Data/tract_shapefiles/tl_2024_06_tract.shp")
KYtract_gdf =  gpd.read_file("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Raw Data/tract_shapefiles/tl_2024_06_tract.shp")
TXtract_gdf =  gpd.read_file("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Raw Data/tract_shapefiles/tl_2024_48_tract.shp")

combined_census_gdf = pd.concat([CAtract_gdf, KYtract_gdf, TXtract_gdf], ignore_index=True)

# projecting ZCTA shapefile to a metric CRS (for distance calculation)
censustract_gdf = combined_census_gdf.to_crs("EPSG:3857")  # Web Mercator: units in meters
restaurants_gdf = gdf.to_crs("EPSG:3857")


In [28]:
# Create a small buffer around each restaurant to avoid precision issues on borders
restaurants_gdf['buffer'] = restaurants_gdf.geometry.buffer(0.001)  # Small buffer

# Perform spatial join with buffered geometries
restaurants_with_tracts = gpd.sjoin(restaurants_gdf, censustract_gdf, how="left", predicate="within")

# Drop duplicates based on the restaurant's geometry
restaurants_with_tracts = restaurants_with_tracts.drop_duplicates(subset='geometry')

# Rename the 'GEOID' column to 'PRIMARY_CENSUSTRACT'
restaurants_with_tracts = restaurants_with_tracts.rename(columns={"GEOID": "PRIMARY_CENSUSTRACT"})

In [29]:
grouped_by_tract = restaurants_with_tracts.groupby('PRIMARY_CENSUSTRACT').size()

# Print the value counts for each ZCTA
print(grouped_by_tract)

PRIMARY_CENSUSTRACT
06037101110     1
06037101122     1
06037101220    13
06037101221     1
06037101222    12
               ..
48491020710     1
48491020810     1
48491020816     3
48491020821     2
48491020822     4
Length: 2423, dtype: int64


In [30]:
#downloading census data
import censusdis.data as ced
from censusdis.states import ALL_STATES_AND_DC

#Variables we want to query
# Variables = ["MEDIAN_HOUSEHOLD_INCOME_VARIABLE", "AVG_HOUSEHOLD_SIZE", "TOTAL_POPULATION", "MEDIAN GROSS RENT"]
VARIABLES = ["NAME","B19013_001E", "B25010_001E", "B01003_001E", "B25064_001E"]

ACS2023_indicators = ced.download(
    dataset="acs/acs5",
    vintage=2023,
    download_variables=VARIABLES,
    state=ALL_STATES_AND_DC,
    tract="*",
    with_geometry=True
)

In [31]:
#merging into zcta_gdf
ACS2023_indicators = ACS2023_indicators.rename(columns={"B19013_001E": "MEDIAN_INCOME", "B25010_001E": "AVG_HOUSEHOLD_SIZE", "B01003_001E": "TOTAL_POPULATION", "B25064_001E": "MEDIAN_GROSS_RENT"})
censustract_gdf = censustract_gdf.merge(ACS2023_indicators[['TRACT', 'MEDIAN_INCOME', 'AVG_HOUSEHOLD_SIZE', 'TOTAL_POPULATION', 'MEDIAN_GROSS_RENT']], left_on="TRACTCE", right_on="TRACT", how="left")

In [ ]:
#For each restaurant, find nearby ZCTAs within 2 miles and average
from shapely.geometry import Point

# Remove any existing 'index_right' column to prevent conflict
if 'index_right' in restaurants_with_tracts.columns:
    restaurants_with_tracts = restaurants_with_tracts.drop(columns='index_right')

# Convert 2 miles to meters
buffer_radius = 3218.69  # meters

# Create buffer around each restaurant point
restaurants_with_tracts['buffer'] = restaurants_with_tracts.geometry.buffer(buffer_radius)

# Use spatial join: buffered restaurant areas to intersecting TRACTS
buffered_join = gpd.sjoin(
    gpd.GeoDataFrame(restaurants_with_tracts, geometry='buffer'),
    censustract_gdf[['TRACT', 'geometry', 'MEDIAN_INCOME', 'AVG_HOUSEHOLD_SIZE', 'TOTAL_POPULATION', 'MEDIAN_GROSS_RENT']],
    how='left',
    predicate='intersects'
)

buffered_join = buffered_join.reset_index()

# Group by restaurant and calculate average income of surrounding ZCTAs
avg_censusindicators_nearby = buffered_join.groupby('index').agg(
    AVG_INCOME_NEARBY_TRACTS=('MEDIAN_INCOME', 'mean'),
    AVG_HH_SIZE_NEARBY_TRACTS=('AVG_HOUSEHOLD_SIZE', 'mean'),
    AVG_RENT_NEARBY_TRACTS=('MEDIAN_GROSS_RENT', 'mean'),
    AVG_POP_NEARBY_TRACTS=('TOTAL_POPULATION', 'mean')
)

# Merge the average income back into the original restaurant dataframe
restaurants_final = restaurants_with_tracts.join(avg_censusindicators_nearby)


In [ ]:
print(restaurants_final.columns.tolist())

['INSPECTION_DATE', 'STORE_NAME', 'STREET_ADDRESS', 'CITY', 'ZIP5', 'SERVICE_DESCRIPTION', 'SCORE', 'GRADE', 'STREET_ADDRESS_LINE2', 'INSPDATE_YEAR', 'STATE', 'Latitude', 'Longitude', 'Accuracy Score', 'Accuracy Type', 'Number', 'Street', 'Unit Type', 'Unit Number', 'City', 'State', 'County', 'Zip', 'Country', 'Source', 'geometry', 'buffer', 'STATEFP', 'COUNTYFP', 'TRACTCE', 'PRIMARY_CENSUSTRACT', 'GEOIDFQ', 'NAME', 'NAMELSAD', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'AVG_INCOME_NEARBY_TRACTS', 'AVG_HH_SIZE_NEARBY_TRACTS', 'AVG_RENT_NEARBY_TRACTS', 'AVG_POP_NEARBY_TRACTS']


In [ ]:
restaurants_final.head()

,INSPECTION_DATE,STORE_NAME,STREET_ADDRESS,CITY,ZIP5,SERVICE_DESCRIPTION,SCORE,GRADE,STREET_ADDRESS_LINE2,INSPDATE_YEAR,...,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,AVG_INCOME_NEARBY_TRACTS,AVG_HH_SIZE_NEARBY_TRACTS,AVG_RENT_NEARBY_TRACTS,AVG_POP_NEARBY_TRACTS
0,2024-01-02,3RD ST CHEVRON,3817 W 3RD ST,LOS ANGELES,90020,ROUTINE INSPECTION,90.0,A,NaN,2024,...,G5020,S,143939.0,0.0,+34.0709091,-118.2984928,65673.235714,2.433571,1553.821429,3419.971429
1,2024-01-02,7-ELEVEN #34473B,5905 SOUTH ST,LAKEWOOD,90713,ROUTINE INSPECTION,97.0,A,NaN,2024,...,G5020,S,1532425.0,0.0,+33.8634660,-118.1171020,99475.972222,2.993889,1970.114286,4567.583333
2,2024-01-02,7-ELEVEN #39023B,4111 VENICE BLVD,LOS ANGELES,90019,ROUTINE INSPECTION,96.0,A,NaN,2024,...,G5020,S,598606.0,0.0,+34.0443254,-118.3275531,80004.714286,2.524732,1518.568807,3574.714286
3,2024-01-02,7-ELEVEN #39714,6051 HOLLYWOOD BLVD # 111,LOS ANGELES,90028,ROUTINE INSPECTION,94.0,A,NaN,2024,...,G5020,S,648704.0,0.0,+34.1009548,-118.3216769,84958.941860,2.173793,1744.494118,3157.775281
4,2024-01-02,AEIRLOOM-CATCHER,10550 RIVERSIDE DR,NORTH HOLLYWOOD,91602,ROUTINE INSPECTION,92.0,A,NaN,2024,...,G5020,S,391203.0,0.0,+34.1501345,-118.3626374,100077.285714,2.290317,1849.901639,3364.727273


In [ ]:
#Export just the geocoded restaurant dataset with census indicators
output_path = "/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Output/healthinspections_2024_censusindicators_censustract.csv"
restaurants_final.to_csv(output_path, index=False)

# Adding SVI data at the Tract level


In [ ]:
#loading CDC Social Vulnerability Index data that is downloaded at the Census Tract level
states_of_interest = ['CA', 'KY', 'TX']

svitract_df = pd.read_csv("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Output/SVI_2022_US.csv")
svitract_df = svitract_df[svitract_df['ST_ABBR'].isin(states_of_interest)]
#add leading zero to geoids where that was dropped by excel format
svitract_df['FIPS'] = svitract_df['FIPS'].astype(str).str.zfill(11)
svitract_df['FIPS'] = svitract_df['FIPS'].astype(str)




In [ ]:
print(svitract_df.columns.tolist())

['ST', 'STATE', 'ST_ABBR', 'STCNTY', 'COUNTY', 'FIPS', 'LOCATION', 'AREA_SQMI', 'E_TOTPOP', 'M_TOTPOP', 'E_HU', 'M_HU', 'E_HH', 'M_HH', 'E_POV150', 'M_POV150', 'E_UNEMP', 'M_UNEMP', 'E_HBURD', 'M_HBURD', 'E_NOHSDP', 'M_NOHSDP', 'E_UNINSUR', 'M_UNINSUR', 'E_AGE65', 'M_AGE65', 'E_AGE17', 'M_AGE17', 'E_DISABL', 'M_DISABL', 'E_SNGPNT', 'M_SNGPNT', 'E_LIMENG', 'M_LIMENG', 'E_MINRTY', 'M_MINRTY', 'E_MUNIT', 'M_MUNIT', 'E_MOBILE', 'M_MOBILE', 'E_CROWD', 'M_CROWD', 'E_NOVEH', 'M_NOVEH', 'E_GROUPQ', 'M_GROUPQ', 'EP_POV150', 'MP_POV150', 'EP_UNEMP', 'MP_UNEMP', 'EP_HBURD', 'MP_HBURD', 'EP_NOHSDP', 'MP_NOHSDP', 'EP_UNINSUR', 'MP_UNINSUR', 'EP_AGE65', 'MP_AGE65', 'EP_AGE17', 'MP_AGE17', 'EP_DISABL', 'MP_DISABL', 'EP_SNGPNT', 'MP_SNGPNT', 'EP_LIMENG', 'MP_LIMENG', 'EP_MINRTY', 'MP_MINRTY', 'EP_MUNIT', 'MP_MUNIT', 'EP_MOBILE', 'MP_MOBILE', 'EP_CROWD', 'MP_CROWD', 'EP_NOVEH', 'MP_NOVEH', 'EP_GROUPQ', 'MP_GROUPQ', 'EPL_POV150', 'EPL_UNEMP', 'EPL_HBURD', 'EPL_NOHSDP', 'EPL_UNINSUR', 'SPL_THEME1', 'RPL_

In [ ]:
#left outerjoin with restaurant data
restaurants_withsvi = restaurants_final.merge(svitract_df, left_on="PRIMARY_CENSUSTRACT", right_on="FIPS", how="left")


In [ ]:
print(restaurants_withsvi.columns.tolist())

['INSPECTION_DATE', 'STORE_NAME', 'STREET_ADDRESS', 'CITY', 'ZIP5', 'SERVICE_DESCRIPTION', 'SCORE', 'GRADE', 'STREET_ADDRESS_LINE2', 'INSPDATE_YEAR', 'STATE_x', 'Latitude', 'Longitude', 'Accuracy Score', 'Accuracy Type', 'Number', 'Street', 'Unit Type', 'Unit Number', 'City', 'State', 'County', 'Zip', 'Country', 'Source', 'geometry', 'buffer', 'STATEFP', 'COUNTYFP', 'TRACTCE', 'PRIMARY_CENSUSTRACT', 'GEOIDFQ', 'NAME', 'NAMELSAD', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'AVG_INCOME_NEARBY_TRACTS', 'AVG_HH_SIZE_NEARBY_TRACTS', 'AVG_RENT_NEARBY_TRACTS', 'AVG_POP_NEARBY_TRACTS', 'ST', 'STATE_y', 'ST_ABBR', 'STCNTY', 'COUNTY', 'FIPS', 'LOCATION', 'AREA_SQMI', 'E_TOTPOP', 'M_TOTPOP', 'E_HU', 'M_HU', 'E_HH', 'M_HH', 'E_POV150', 'M_POV150', 'E_UNEMP', 'M_UNEMP', 'E_HBURD', 'M_HBURD', 'E_NOHSDP', 'M_NOHSDP', 'E_UNINSUR', 'M_UNINSUR', 'E_AGE65', 'M_AGE65', 'E_AGE17', 'M_AGE17', 'E_DISABL', 'M_DISABL', 'E_SNGPNT', 'M_SNGPNT', 'E_LIMENG', 'M_LIMENG', 'E_MINRTY', 'M_MINRTY',

In [ ]:
#drop unneeded columns
svicols_todrop = ["ST", "STATE_x", "STATE_y", "STCNTY", "COUNTY", "FIPS", "M_TOTPOP", "M_HU", "M_HH", "M_POV150",
                  'M_UNEMP',  'M_HBURD', 'M_NOHSDP', 'M_UNINSUR', 'M_AGE65', 'M_AGE17', 
                  'M_DISABL', 'M_SNGPNT', 'M_LIMENG',  'M_MINRTY','M_MUNIT', 'M_MOBILE', 
                  'M_CROWD', 'M_NOVEH', 'M_GROUPQ', 'MP_POV150', 'MP_UNEMP', 
                  'MP_HBURD', 'MP_NOHSDP', 'MP_UNINSUR', 'MP_AGE65', 'MP_AGE17', 
                  'MP_DISABL', 'MP_SNGPNT', 'MP_LIMENG', 'MP_MINRTY', 'MP_MUNIT',
                    'MP_MOBILE', 'MP_CROWD', 'MP_NOVEH', 'MP_GROUPQ', 'M_NOINT', 
                    'M_AFAM', 'M_HISP', 'M_ASIAN', 'M_AIAN', 'M_NHPI', 'M_TWOMORE', 
                    'M_OTHERRACE','MP_NOINT', 'MP_AFAM', 'MP_HISP', 'MP_ASIAN', 
                    'MP_AIAN', 'MP_NHPI', 'MP_TWOMORE', 'MP_OTHERRACE']
restaurants_withsvi = restaurants_withsvi.drop(columns=svicols_todrop)

In [ ]:
#Exporting census tract level data with svi
restaurants_withsvi.to_csv("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Output/healthinspections_2024_withsvi_censustract", index=False)

# Merging Food Access Research Atlas Data

In [ ]:
usda_foodaccessdf = pd.read_excel("Output/FoodAccessResearchAtlasData2019.xlsx", sheet_name=2)
states_needed = ['California', 'Kentucky', 'Texas']
usda_foodaccessdf = usda_foodaccessdf[usda_foodaccessdf['State'].isin(states_needed)]

In [ ]:
usda_colstodrop = ["State", "County"]
usda_foodaccessdf = usda_foodaccessdf.drop(columns=usda_colstodrop)
#add leading zero to geoids where that was dropped by excel format
usda_foodaccessdf['CensusTract'] = usda_foodaccessdf['CensusTract'].astype(str).str.zfill(11)
usda_foodaccessdf['CensusTract'] = usda_foodaccessdf['CensusTract'].astype(str)
usda_foodaccessdf = usda_foodaccessdf.rename(columns=lambda col: f"USDA_{col}")


In [ ]:
#merge USDA Food Access data with restaurant dataset
restaurants_withsvifoodaccess = restaurants_withsvi.merge(usda_foodaccessdf, left_on="PRIMARY_CENSUSTRACT", right_on="USDA_CensusTract", how="left")


In [ ]:
#export data
output_path_final = "/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Output/final_withsvicensusandUSDAfoodaccess.csv"
restaurants_withsvifoodaccess.to_csv(output_path_final, index=False)

In [ ]:
#count by state
state_count = restaurants_withsvifoodaccess["STATEFP"].value_counts()
print(state_count)

STATEFP
06    22415
48     3026
Name: count, dtype: int64


In [ ]:
state_count2 = restaurants_withsvifoodaccess["State"].value_counts()
print(state_count2)

KeyError: 'State_x'